# Day 5 — Project build

In [1]:
import pandas as pd

raw = pd.read_csv("day4_raw_employee_attrition.csv")
print(raw.shape)
raw.head()

(300, 10)


,employee_id,age,monthly_income,years_at_company,job_satisfaction,distance_from_home_km,overtime,department,attrition,exit_interview_done
0,1,25,69277.0,10,4,4.2,No,Sales,0,No
1,2,46,53469.0,12,2,9.5,No,Sales,0,No
2,3,24,56296.0,12,4,2.3,Yes,R&D,0,No
3,4,40,44986.0,19,3,6.0,No,Sales,0,No
4,5,44,52345.0,3,2,0.9,No,Sales,0,No


In [2]:
clean = raw.copy()

# 1) drop the leaky column found yesterday - it wouldn't exist at prediction time
clean = clean.drop(columns=["exit_interview_done"])

# 2) handle missing monthly_income -> fill with median (same reasoning as Day 3)
clean["monthly_income"] = clean["monthly_income"].fillna(clean["monthly_income"].median())

# 3) handle the outliers found on Day 4 -> recompute IQR bounds, then cap instead of deleting rows
q1 = clean["monthly_income"].quantile(0.25)
q3 = clean["monthly_income"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
clean["monthly_income"] = clean["monthly_income"].clip(lower=lower_bound, upper=upper_bound)

# 4) quick check - confirming no more missing values and outliers are reined in
print("Missing values left:")
print(clean.isna().sum())
print()
print("Monthly income range after cleaning:", clean["monthly_income"].min(), "-", clean["monthly_income"].max())

Missing values left:
employee_id              0
age                      0
monthly_income           0
years_at_company         0
job_satisfaction         0
distance_from_home_km    0
overtime                 0
department               0
attrition                0
dtype: int64

Monthly income range after cleaning: 19208.875 - 87889.875


In [3]:
# saving the cleaned dataset so it's a genuine reusable artifact, not just something living in this notebook
clean.to_csv("day5_clean_employee_attrition.csv", index=False)
print("Saved -> day5_clean_employee_attrition.csv")
clean.head()

Saved -> day5_clean_employee_attrition.csv


,employee_id,age,monthly_income,years_at_company,job_satisfaction,distance_from_home_km,overtime,department,attrition
0,1,25,69277.0,10,4,4.2,No,Sales,0
1,2,46,53469.0,12,2,9.5,No,Sales,0
2,3,24,56296.0,12,4,2.3,Yes,R&D,0
3,4,40,44986.0,19,3,6.0,No,Sales,0
4,5,44,52345.0,3,2,0.9,No,Sales,0


### 📖 Data dictionary

Writing this out properly because "what does this column even mean" is exactly the kind of question I'd forget the answer to a week from now.

| Column | Type | Description |
|---|---|---|
| `employee_id` | int | unique id per employee, not a real feature |
| `age` | int | employee age in years |
| `monthly_income` | float | monthly income; cleaned for missing values + capped outliers |
| `years_at_company` | int | tenure in years |
| `job_satisfaction` | int (1-4) | self-reported satisfaction, 1 = low, 4 = high |
| `distance_from_home_km` | float | commute distance in km |
| `overtime` | Yes/No | whether employee regularly works overtime |
| `department` | category | Sales / R&D / HR |
| `attrition` | 0/1 (target) | 1 = employee left, 0 = stayed |
| ~~`exit_interview_done`~~ | — | **dropped** — leaks the target, only exists after someone leaves |

### Missing & outlier summary
- `monthly_income`: ~20/300 rows were missing → filled with median.
- `monthly_income`: 3 rows were extreme outliers (~6x normal) → capped to the IQR bounds instead of deleting the rows, so I keep the rest of that employee's data.
- No missing values in any other column for this dataset (real datasets probably won't be this convenient — noted for later).
